In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

import teehr
from teehr import RemoteReadWriteEvaluation
from teehr.utilities.apply_migrations import evolve_catalog_schema
from teehr.evaluation.spark_session_utils import create_spark_session
from teehr import Unit, Configuration, Variable
import utils.nrcs_utils as nrcs_utils
import utils.cbrfc_wsv_seasonal_utils as cbrfc_wsv_seasonal_utils

### Setup

In [ ]:
spark = create_spark_session()

In [ ]:
migrations_dir = Path(teehr.__file__).parent / "migrations"

evolve_catalog_schema(
    spark=spark,
    migrations_dir_path=migrations_dir,
    target_catalog_name="iceberg",
    target_namespace_name="teehr"
)

In [ ]:
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

In [ ]:
# configure download
ev.download.configure(
    api_key='your_key_here'
)

### Add and populate domain tables

#### Define file paths for the locally-stored locations tables

In [ ]:
locations_path = Path(Path.cwd(), "locations", "locations.csv")
location_crosswalks_path = Path(Path.cwd(), "locations", "crosswalk.csv")
location_aliases_path = Path(Path.cwd(), "locations", "aliases.csv")

#### Pull domain tables from teehr warehouse

In [ ]:
print("Loading domain tables.")

print("Loading configurations.")
ev.download.configurations(
    load=True
)

print("Loading units.")
ev.download.units(
    load=True
)

print("Loading variables.")
ev.download.variables(
    load=True
)

print("Loading locations.")
df = pd.read_csv(locations_path)
gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkt(df["geometry"]), crs="EPSG:4326")
ev.locations.load_dataframe(df=gdf, write_mode="upsert")

print("Loading location crosswalks.")
df = pd.read_csv(location_crosswalks_path)
ev.location_crosswalks.load_dataframe(
    df=df,
    write_mode="upsert"
)

print("Loading location aliases.")
df = pd.read_csv(location_aliases_path)
ev.location_id_aliases.load_dataframe(
    df=df,
    write_mode="upsert"
)

print("Loading attributes.")
ev.download.attributes(
    load=True
)

'''
# skipping -- must connect to FVED internal-ids

print("Loading location attributes.")
sdf = ev.location_id_aliases.filter("location_id_alias LIKE 'usgs-%'").to_sdf()
usgs_locs = [row['location_id_alias'] for row in sdf.collect()]
ev.download.location_attributes(
    location_id=usgs_locs,
    load=True
)
'''

print("Loading domain variables, locations, crosswalks, and attributes complete!")

#### Add general domain entries to support FVED data ingest

In [ ]:
# add unit for volume
unit = Unit(
    name='m^3',
    long_name='Cubic Meter'
)
ev.units.add(unit)

#### Add domain entries to support cbrfc_unregulated_esp_forecast configuration data

In [ ]:
# add configurations for cbrfc wsv's (use existing for usgs/nwm)
configuration = Configuration(
    name="cbrfc_unregulated_esp_forecast",
    timeseries_type="secondary",
    description="CBRFC unregulated ESP water supply volume forecasts",
)
ev.configurations.add(configuration)

# add variable for monthly wsv's
variable = Variable(
    name="volume_monthly_sum",
    long_name="Monthly summed water supply volume"
)
ev.variables.add(variable)

#### Add domain entries to support cbrfc_unregulated_official_forecast configuration data (includes observed)

In [ ]:
# add configurations for cbrfc seasonal wsv data to the configurations table
configuration = Configuration(
    name="cbrfc_unregulated_official_forecast",
    timeseries_type="secondary",
    description="CBRFC unregulated official water supply volume forecasts",
)
ev.configurations.add(configuration)
configuration = Configuration(
    name="cbrfc_unregulated_observed",
    timeseries_type="primary",
    description="CBRFC unregulated water-supply volume observations",
)
ev.configurations.add(configuration)

# add variables for seasonal wsv data using mapping from cbrfc_wsv_seasonal_utils
# these were the three unique cases for CBRFC
for key in cbrfc_wsv_seasonal_utils.CBRFC_SEASONAL_VARIABLE_MAPPING.keys():
    variable = Variable(
        name=key,
        long_name=cbrfc_wsv_seasonal_utils.CBRFC_SEASONAL_VARIABLE_MAPPING[key],
    )
    ev.variables.add(variable)

#### Add domain entries to support nrcs_adjusted_forecast configurations data (includes observed)

In [ ]:
# add configuration for nrcs_adjusted_forecast and observed
configuration = Configuration(
    name="nrcs_adjusted_forecast",
    timeseries_type="secondary",
    description="NRCS adjusted water supply volume forecasts",
)
ev.configurations.add(configuration)
configuration = Configuration(
    name="nrcs_adjusted_observed",
    timeseries_type="primary",
    description="NRCS adjusted water supply volume observations",
)
ev.configurations.add(configuration)

# add variables for nrcs_adjusted_forecast using mapping from nrcs_utils
# Note: observed variable satisfied when loading the cbrfc_esp data
for key in nrcs_utils.NRCS_VARIABLE_MAPPING.keys():
    variable = Variable(
        name=key,
        long_name=nrcs_utils.NRCS_VARIABLE_MAPPING[key],
    )
    ev.variables.add(variable)

#### Add domain entries to support cdec_forecast configuration data (includes observed)

In [ ]:
# add configurations for cdec water supply forecasts and observations (ingested by the ingest-cdec-water-supply flow)
configuration = Configuration(
    name="cdec_forecast",
    timeseries_type="secondary",
    description="CDEC forecasts",
)
ev.configurations.add(configuration)
configuration = Configuration(
    name="cdec_observed",
    timeseries_type="primary",
    description="CDEC observations",
)
ev.configurations.add(configuration)

# add variables for cdec seasonal volumes, named per the FVED schema conventions
# volume_monthly_sum is added above
ev.variables.add([
    Variable(name="volume_0407_sum", long_name="Seasonal summed water supply volume (Apr-Jul)"),
    Variable(name="volume_1009_sum", long_name="Seasonal summed water supply volume (Oct-Sep)"),
    Variable(name="volume_0407_cumsum", long_name="Cumulative water supply volume (Apr-Jul) at each month end"),
])

#### Add configuration_pairs table entries

In [ ]:
primaries = []
secondaries = []

### Kill spark

In [ ]:
spark.stop()